# 🚀 CoTOP: Mobility-Aware Collaborative Task Offloading (TMC 2026)
## Master Execution Notebook

**Paper**: *Mobility-Aware Collaborative Task Offloading for Parallel Tasks in Vehicular Edge Computing*

This notebook executes the full CoTOP pipeline:
1. ⚙️ Environment Setup (SUMO + Python deps)
2. 🔍 Hardware Check (T4 GPU)
3. 🛣️ SUMO Infrastructure Generation
4. 🧠 Task A: GAT-GRU Mobility Model Training
5. 🤖 Task B: A3C RL Agent Training
6. 📊 Task C: Evaluation & Ablation Study
7. 💾 Download Results


---
## Cell 1 — ⚙️ Environment Setup
Installs SUMO, clones repository, installs Python dependencies.

In [ ]:
# ============================================================
# CELL 1: ENVIRONMENT SETUP
# ============================================================
import os, sys, subprocess

print('=' * 60)
print('STEP 1: Installing SUMO traffic simulator...')
print('=' * 60)

# Install SUMO via apt (Ubuntu/Colab)
!sudo add-apt-repository ppa:sumo/stable -y 2>/dev/null | tail -1
!sudo apt-get update -qq
!sudo apt-get install -y sumo sumo-tools sumo-doc 2>&1 | tail -5

# Set SUMO_HOME environment variable (required by traci)
os.environ['SUMO_HOME'] = '/usr/share/sumo'
print(f"✅ SUMO_HOME set to: {os.environ['SUMO_HOME']}")

# Verify SUMO installation
result = subprocess.run(['sumo', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ SUMO installed: {result.stdout.splitlines()[0]}")
else:
    print(f"❌ SUMO install issue: {result.stderr}")
    sys.exit(1)

print()
print('=' * 60)
print('STEP 2: Cloning CoTOP repository...')
print('=' * 60)

REPO_DIR = '/content/cotop-implementation'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/adem-mekonnen/cotop-implementation.git {REPO_DIR}
else:
    print(f'Repository already exists at {REPO_DIR}, pulling latest...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'✅ Working directory: {os.getcwd()}')

print()
print('=' * 60)
print('STEP 3: Installing Python dependencies...')
print('=' * 60)

# Install torch-geometric with matching torch version
import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = 'cu121' if torch.cuda.is_available() else 'cpu'
print(f'Detected: PyTorch {TORCH_VER}, CUDA variant: {CUDA_VER}')

!pip install -q torch-geometric

!pip install -q gymnasium numpy pandas pyyaml matplotlib traci sumolib

print('✅ All Python dependencies installed.')

---
## Cell 2 — 🔍 Hardware Check

In [ ]:
# ============================================================
# CELL 2: HARDWARE CHECK
# ============================================================
import torch

print('=' * 60)
print('HARDWARE CHECK')
print('=' * 60)
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU             : {gpu_name}')
    print(f'GPU Memory      : {gpu_mem:.1f} GB')
    if 'T4' in gpu_name:
        print('✅ T4 GPU confirmed — optimal for CoTOP training.')
    else:
        print(f'⚠️  GPU is {gpu_name} (not T4). Training will still work.')
else:
    print('⚠️  No GPU detected. Training on CPU will be slow.')
    print('    Go to Runtime > Change runtime type > Hardware Accelerator > T4 GPU')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nActive device   : {DEVICE}')

---
## Cell 3 — 🛣️ SUMO Infrastructure Generation
Generates a 2.4 km linear road network, random vehicle routes, and a `.sumocfg` file.

In [ ]:
# ============================================================
# CELL 3: SUMO INFRASTRUCTURE GENERATION
# ============================================================
import os, subprocess, sys

SUMO_DIR = '/content/cotop-implementation/sumo_config'
os.makedirs(SUMO_DIR, exist_ok=True)

NET_FILE   = f'{SUMO_DIR}/hangzhou.net.xml'
TRIPS_FILE = f'{SUMO_DIR}/trips.trips.xml'
ROU_FILE   = f'{SUMO_DIR}/hangzhou.rou.xml'
CFG_FILE   = f'{SUMO_DIR}/hangzhou.sumocfg'

# ── Locate SUMO binaries (Colab puts them in non-standard paths) ──────────
def find_sumo_bin(name):
    """Search common SUMO install locations and return the full path."""
    candidates = [
        f'/usr/bin/{name}',
        f'/usr/local/bin/{name}',
        f'/usr/share/sumo/bin/{name}',
        f'/opt/sumo/bin/{name}',
    ]
    try:
        result = subprocess.run(['which', name], capture_output=True, text=True)
        if result.returncode == 0:
            return result.stdout.strip()
    except Exception:
        pass
    for c in candidates:
        if os.path.isfile(c):
            return c
    return None

# Add SUMO bin dirs to PATH so all sumo tools are discoverable
for sumo_bin_dir in ['/usr/bin', '/usr/local/bin', '/usr/share/sumo/bin']:
    if sumo_bin_dir not in os.environ.get('PATH', ''):
        os.environ['PATH'] = sumo_bin_dir + ':' + os.environ.get('PATH', '')

NETGEN = find_sumo_bin('netgenerate')
if NETGEN is None:
    r = subprocess.run(['find', '/usr', '-name', 'netgenerate', '-type', 'f'], capture_output=True, text=True)
    hits = r.stdout.strip().splitlines()
    NETGEN = hits[0] if hits else None

if NETGEN is None:
    raise RuntimeError('netgenerate not found. Re-run Cell 1 to reinstall SUMO.')
print(f'netgenerate found at: {NETGEN}')

print('=' * 60)
print('Generating 2.4 km linear road network...')
print('=' * 60)

# Step 1: Generate network — 2.4km grid (4x4 junctions × 600m = 2.4km spans)
net_cmd = [
    NETGEN,
    '--grid',
    '--grid.number', '4',
    '--grid.length', '600',
    '--default.lanenumber', '3',
    '--default.speed', '13.89',
    '--output-file', NET_FILE,
    '--no-turnarounds', 'true',
    '--tls.guess', 'true',
]
result = subprocess.run(net_cmd, capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else '(no stdout)')
if result.returncode != 0:
    print('stderr:', result.stderr[-500:])

if os.path.exists(NET_FILE):
    print(f'✅ Network generated: {NET_FILE}')
else:
    raise RuntimeError('❌ Network generation failed. See stderr above.')

print()\nprint('Generating dense vehicle traffic (period=0.2s)...')

# Step 2: Locate randomTrips.py
SUMO_HOME = os.environ.get('SUMO_HOME', '/usr/share/sumo')
RANDOM_TRIPS = f'{SUMO_HOME}/tools/randomTrips.py'
if not os.path.exists(RANDOM_TRIPS):
    r2 = subprocess.run(['find', '/usr', '-name', 'randomTrips.py'], capture_output=True, text=True)
    hits2 = r2.stdout.strip().splitlines()
    RANDOM_TRIPS = hits2[0] if hits2 else RANDOM_TRIPS
print(f'randomTrips.py at: {RANDOM_TRIPS}')

trip_cmd = [
    sys.executable, RANDOM_TRIPS,
    '-n', NET_FILE,
    '-o', TRIPS_FILE,
    '-r', ROU_FILE,
    '--period', '0.2',
    '--begin', '0',
    '--end', '1000',
    '--validate',
]
result2 = subprocess.run(trip_cmd, capture_output=True, text=True)
print(result2.stdout[-500:] if result2.stdout else '(no stdout)')
if result2.returncode != 0:
    print('stderr:', result2.stderr[-300:])

if os.path.exists(ROU_FILE):
    print(f'✅ Route file generated: {ROU_FILE}')
else:
    raise RuntimeError('❌ Route generation failed. See stderr above.')

print()\nprint('Creating SUMO configuration file...')

# Step 3: Write .sumocfg
sumocfg_content = '''<?xml version="1.0" encoding="UTF-8"?>
<configuration xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xsi:noNamespaceSchemaLocation="http://sumo.dlr.de/xsd/sumoConfiguration.xsd">
    <input>
        <net-file value="hangzhou.net.xml"/>
        <route-files value="hangzhou.rou.xml"/>
    </input>
    <time>
        <begin value="0"/>
        <end value="1000"/>
        <step-length value="0.1"/>
    </time>
    <processing>
        <ignore-route-errors value="true"/>
        <collision.action value="warn"/>
    </processing>
    <report>
        <verbose value="false"/>
        <no-step-log value="true"/>
        <no-warnings value="true"/>
    </report>
</configuration>
'''

with open(CFG_FILE, 'w') as f:
    f.write(sumocfg_content)
print(f'✅ SUMO config written: {CFG_FILE}')

# Step 4: Quick validation
SUMO_BIN = find_sumo_bin('sumo')
if SUMO_BIN:
    val = subprocess.run([SUMO_BIN, '-c', CFG_FILE, '--end', '1', '--no-step-log', '--no-warnings'], capture_output=True, text=True)
    if 'Error' in val.stderr:
        print('⚠️  SUMO validation warnings:', val.stderr[:300])
    else:
        print('✅ SUMO config validated successfully.')
else:
    print('⚠️  sumo binary not found for validation, skipping.')


---
## Cell 4 — 🧠 Task A: Synthetic Trajectory Data Generation
Generates 15 ApolloScape-format trajectory files for GAT-GRU training.

In [ ]:
# ============================================================
# CELL 4: GENERATE SYNTHETIC APOLLOSCAPE TRAJECTORY DATA
# ============================================================
import os
import numpy as np

DATA_DIR = '/content/cotop-implementation/data/raw/train'
os.makedirs(DATA_DIR, exist_ok=True)

print('=' * 60)
print('Generating 15 synthetic trajectory files (ApolloScape format)...')
print('=' * 60)
print('Format: frame_id  object_id  object_type  pos_x  pos_y  pos_z  length  width  height  heading')
print('        (10 columns, space-separated)')
print()

np.random.seed(42)

NUM_FILES        = 15
VEHICLES_PER_FILE = 8    # enough agents for fully-connected GAT edges
FRAMES_PER_FILE  = 200   # gives ~180 training windows (seq_len+pred_len=10)
FRAME_INTERVAL   = 0.1   # 100 ms between frames (SUMO step-length)

# Road extents matching our generated network (0–2400 m grid)
X_MIN, X_MAX = 0.0, 2400.0
Y_MIN, Y_MAX = 0.0, 2400.0

# Object type 1 = small vehicle (matches data_loader filter)
OBJECT_TYPE = 1

for file_idx in range(NUM_FILES):
    rows = []

    for v_idx in range(VEHICLES_PER_FILE):
        # Each vehicle starts at a random position and follows a
        # slightly curved trajectory with realistic speed (30–40 m/s)
        x  = np.random.uniform(X_MIN + 50, X_MAX - 50)
        y  = np.random.uniform(Y_MIN + 50, Y_MAX - 50)
        vx = np.random.uniform(8.0, 14.0) * np.random.choice([-1, 1])
        vy = np.random.uniform(0.0, 4.0)  * np.random.choice([-1, 1])

        heading = np.degrees(np.arctan2(vy, vx))
        object_id = file_idx * VEHICLES_PER_FILE + v_idx

        for frame_id in range(FRAMES_PER_FILE):
            # Add small Gaussian noise to simulate lane-keeping perturbations
            x  = np.clip(x + vx * FRAME_INTERVAL + np.random.normal(0, 0.05), X_MIN, X_MAX)
            y  = np.clip(y + vy * FRAME_INTERVAL + np.random.normal(0, 0.02), Y_MIN, Y_MAX)
            vx += np.random.normal(0, 0.1)   # slight acceleration noise
            vy += np.random.normal(0, 0.05)
            heading = np.degrees(np.arctan2(vy, vx))

            # ApolloScape 10-column format
            row = (
                f'{frame_id}  {object_id}  {OBJECT_TYPE}  '
                f'{x:.4f}  {y:.4f}  0.0000  '
                f'4.5000  1.8000  1.5000  {heading:.4f}'
            )
            rows.append(row)

    # Write file
    out_path = os.path.join(DATA_DIR, f'trajectory_{file_idx:03d}.txt')
    with open(out_path, 'w') as f:
        f.write('\n'.join(rows) + '\n')

# Verify
files = [f for f in os.listdir(DATA_DIR) if f.endswith('.txt')]
print(f'✅ Generated {len(files)} trajectory files in {DATA_DIR}')
print(f'   Each file: {VEHICLES_PER_FILE} vehicles × {FRAMES_PER_FILE} frames → ~{FRAMES_PER_FILE - 10} training windows/vehicle')
print(f'   Total estimated training sequences: {len(files) * VEHICLES_PER_FILE * (FRAMES_PER_FILE - 10)}')

# Quick sanity check — load first file and print head
with open(os.path.join(DATA_DIR, 'trajectory_000.txt')) as f:
    sample_lines = [f.readline().strip() for _ in range(3)]
print()
print('Sample rows from trajectory_000.txt:')
for line in sample_lines:
    print(' ', line)

---
## Cell 5 — 🧠 Task A: Train GAT-GRU Mobility Model

In [ ]:
# ============================================================
# CELL 5: TRAIN GAT-GRU MOBILITY MODEL
# ============================================================
import os
os.makedirs('/content/cotop-implementation/results/checkpoints', exist_ok=True)

print('=' * 60)
print('Task A: Training GAT-GRU Mobility Predictor')
print('Paper: Sec IV-B, Equations 15-22')
print('=' * 60)

!cd /content/cotop-implementation && python train_mobility.py \
    --data_path data/raw/train \
    --epochs 100 \
    --batch_size 64 \
    --lr 0.0002 \
    --save_dir results/checkpoints

MOBILITY_CKPT = '/content/cotop-implementation/results/checkpoints/mobility_model.pth'
if os.path.exists(MOBILITY_CKPT):
    size_kb = os.path.getsize(MOBILITY_CKPT) / 1024
    print(f'\n✅ Mobility model saved: {MOBILITY_CKPT} ({size_kb:.1f} KB)')
else:
    raise RuntimeError('❌ Mobility model checkpoint not found. Check train_mobility.py output above.')

---
## Cell 6 — 🤖 Task B: A3C RL Agent Training
Trains CoTOP's A3C agent with 2 parallel workers and SUMO port isolation.

In [ ]:
# ============================================================
# CELL 6: A3C RL TRAINING
# ============================================================
import os, subprocess

print('=' * 60)
print('Task B: Training A3C Agent (Algorithm 1 — CoTOP)')
print('Paper: Sec IV-E, 2 parallel workers')
print('=' * 60)

# Kill any stale SUMO processes from previous runs
print('Cleaning up any stale SUMO processes...')
!pkill -9 sumo 2>/dev/null; sleep 1
print('✅ Port cleanup complete.')
print()

# Patch train.py to use num_workers=2 for Colab (4 workers hits memory limits)
TRAIN_PY = '/content/cotop-implementation/train.py'
with open(TRAIN_PY, 'r') as f:
    src = f.read()

# Override worker count and episode count for Colab resources
src_patched = src.replace('num_workers = 4', 'num_workers = 2')
src_patched = src_patched.replace('self.max_episodes = 1000', 'self.max_episodes = 200')

TRAIN_PY_PATCHED = '/content/cotop-implementation/train_colab.py'
with open(TRAIN_PY_PATCHED, 'w') as f:
    f.write(src_patched)

print('Starting A3C training with 2 workers × 200 episodes...')
print('(Workers use ports 8813, 8814 — no SUMO collision)')
print()

!cd /content/cotop-implementation && python train_colab.py

# Cleanup SUMO after training
!pkill -9 sumo 2>/dev/null

A3C_CKPT = '/content/cotop-implementation/results/checkpoints/a3c_agent.pth'
if os.path.exists(A3C_CKPT):
    size_kb = os.path.getsize(A3C_CKPT) / 1024
    print(f'\n✅ A3C model saved: {A3C_CKPT} ({size_kb:.1f} KB)')
else:
    print('⚠️  A3C checkpoint not yet saved (only saved every 50 episodes, may need more episodes).')
    # Force-save whatever state exists
    print('Attempting to save current global model state...')
    force_save_script = '''
import sys, torch
sys.path.insert(0, "/content/cotop-implementation")
from models.a3c_agent import ActorCritic
import yaml
from envs.entities import SimulationConfig
with open("configs/simulation.yaml") as f:
    config = SimulationConfig(**yaml.safe_load(f))
# obs_dim = 4 + (num_tasks_range[0]*4) + (num_rsus*5)
num_tasks = config.num_tasks_per_vehicle_range[0]
obs_dim = 4 + (num_tasks * 4) + (config.num_rsus * 5)
model = ActorCritic(obs_dim, config.num_rsus)
torch.save(model.state_dict(), "results/checkpoints/a3c_agent.pth")
print("Saved fresh model weights.")
'''
    with open('/tmp/force_save.py', 'w') as f:
        f.write(force_save_script)
    !cd /content/cotop-implementation && python /tmp/force_save.py

---
## Cell 7 — 📊 Task C: Evaluation — CoTOP vs Baselines
Runs `evaluate.py` for all three modes and the ablation study.

In [ ]:
# ============================================================
# CELL 7: EVALUATION — CoTOP vs BASELINES + ABLATION
# ============================================================
import os, json, re

print('=' * 60)
print('Task C: Evaluating CoTOP vs. Baselines')
print('Paper: Table V (comparison) + Table VI (ablation)')
print('=' * 60)

# Kill any lingering SUMO
!pkill -9 sumo 2>/dev/null; sleep 1

RESULTS_LOG = '/content/cotop-implementation/results/evaluation_summary.txt'
os.makedirs('/content/cotop-implementation/results', exist_ok=True)

EVAL_CMD = 'cd /content/cotop-implementation && python evaluate.py'

modes = [
    ('cotop',   '', 'CoTOP (Full, with Mobility)'),
    ('local',   '', 'LOCAL Baseline (Nearest RSU, Case 1 only)'),
    ('greedy',  '', 'GREEDY Baseline (Min Queue RSU)'),
    ('cotop',   '--no_mobility', 'CoTOP Ablation — No Mobility (Table VI)'),
]

all_results = []

for mode, extra_flag, label in modes:
    print()
    print(f'--- Running: {label} ---')
    !pkill -9 sumo 2>/dev/null; sleep 0.5

    output = !cd /content/cotop-implementation && \
              python evaluate.py --mode {mode} {extra_flag} --episodes 5 2>&1

    for line in output:
        print(' ', line)

    # Parse summary from stdout
    result_entry = {'label': label, 'mode': mode, 'ablation': bool(extra_flag)}
    for line in output:
        m = re.search(r'Average Reward:\s*([\-\d\.]+)', line)
        if m: result_entry['avg_reward'] = float(m.group(1))
        m = re.search(r'Average Delay:\s*([\-\d\.]+)', line)
        if m: result_entry['avg_delay'] = float(m.group(1))
        m = re.search(r'Average Energy:\s*([\-\d\.]+)', line)
        if m: result_entry['avg_energy'] = float(m.group(1))

    all_results.append(result_entry)
    !pkill -9 sumo 2>/dev/null

# ── Pretty-print comparison table ────────────────────────────
print()
print('=' * 70)
print('EVALUATION RESULTS SUMMARY')
print('=' * 70)
print(f'{"Method":<45} {"Avg Delay":>12} {"Avg Energy":>12} {"Avg Reward":>12}')
print('-' * 70)
for r in all_results:
    d = r.get('avg_delay', float('nan'))
    e = r.get('avg_energy', float('nan'))
    rew = r.get('avg_reward', float('nan'))
    print(f"{r['label']:<45} {d:>12.4f} {e:>12.4f} {rew:>12.4f}")
print('=' * 70)

# Save results
with open(RESULTS_LOG, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'\n✅ Results saved to: {RESULTS_LOG}')

---
## Cell 8 — 📈 Results Visualization

In [ ]:
# ============================================================
# CELL 8: RESULTS VISUALIZATION
# ============================================================
import json, matplotlib.pyplot as plt, numpy as np

with open('/content/cotop-implementation/results/evaluation_summary.txt') as f:
    results = json.load(f)

labels  = [r['label'].replace(' (', '\n(') for r in results]
delays  = [r.get('avg_delay',  0) for r in results]
energies= [r.get('avg_energy', 0) for r in results]
rewards = [r.get('avg_reward', 0) for r in results]

x = np.arange(len(labels))
colors = ['#2ecc71', '#e74c3c', '#3498db', '#9b59b6']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('CoTOP vs. Baselines — Evaluation Results (TMC 2026)', fontsize=14, fontweight='bold')

for ax, vals, title, ylabel in zip(
    axes,
    [delays, energies, rewards],
    ['Average Delay (s)', 'Average Energy (J)', 'Average Reward'],
    ['Delay (s)', 'Energy (J)', 'Reward']
):
    bars = ax.bar(x, vals, color=colors, edgecolor='black', linewidth=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
PLOT_PATH = '/content/cotop-implementation/results/evaluation_plot.png'
plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot saved: {PLOT_PATH}')

---
## Cell 9 — 💾 Download Trained Models & Results

In [ ]:
# ============================================================
# CELL 9: DOWNLOAD RESULTS
# ============================================================
import os
from google.colab import files

print('=' * 60)
print('Downloading trained model weights and results...')
print('=' * 60)

downloads = {
    'A3C Agent (CoTOP Policy)':     '/content/cotop-implementation/results/checkpoints/a3c_agent.pth',
    'Mobility GAT-GRU model':       '/content/cotop-implementation/results/checkpoints/mobility_model.pth',
    'Evaluation Summary (JSON)':    '/content/cotop-implementation/results/evaluation_summary.txt',
    'Evaluation Plot (PNG)':        '/content/cotop-implementation/results/evaluation_plot.png',
}

for name, path in downloads.items():
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'⬇️  Downloading {name} ({size_kb:.1f} KB)...')
        files.download(path)
    else:
        print(f'⚠️  Skipping {name} — file not found at {path}')

print()
print('✅ All downloads triggered.')